In [ ]:
import os
import json
from pdf2image import convert_from_path
import base64
from io import BytesIO
from PIL import Image
from openai import OpenAI

# ========== CONFIG ==========
MODEL = "gpt-4o-mini"   # or "gpt-5-mini" if you have access
EXTRACTION_PROMPT = """
You are a helpful parser. Extract structured JSON from the following OCR text.
return all data given in pdf 

OCR TEXT:
{OCR_TEXT}
"""
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)
# Set your OpenAI key in environment variable: export OPENAI_API_KEY="your_key"
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ========== OCR FUNCTION ==========
def ocr_pdf(pdf_path):
    """Convert scanned PDF to text using OpenAI Vision model"""
    images = convert_from_path(pdf_path)
    ocr_text = ""
    for i, image in enumerate(images):
        # Convert PIL image to base64
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()
        
        # Use OpenAI Vision model to extract text
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Extract all text from this image. Return only the extracted text without any additional formatting or explanations."
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{img_base64}"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=4000,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"[ERROR] OpenAI Vision OCR failed for page {i+1}: {e}")
            text = ""
        
        ocr_text += f"\n--- Page {i+1} ---\n{text}\n"
    return ocr_text.strip()

# ========== GPT FUNCTION ==========
def call_openai_for_extraction(ocr_text, model=MODEL, max_tokens=800):
    """Send OCR text to GPT and return structured JSON"""
    prompt = EXTRACTION_PROMPT.replace("{OCR_TEXT}", ocr_text)

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful parser that extracts structured data from OCR'd text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=max_tokens
    )

    output = resp.choices[0].message.content.strip()

    try:
        start = output.find("{")
        end = output.rfind("}") + 1
        json_text = output[start:end]
        parsed = json.loads(json_text)
        parsed["_raw_model_output"] = output
        return parsed
    except Exception as e:
        return {"error": "failed_to_parse_model_output", "raw": output, "exception": str(e)}

# ========== MAIN PIPELINE ==========
if __name__ == "__main__":
    pdf_path = "testing ocr.pdf"   # 👈 put your scanned PDF here
    print("📄 Running OCR on:", pdf_path)

    ocr_text = ocr_pdf(pdf_path)
    print("\n✅ OCR Extracted Text (first 500 chars):\n", ocr_text[:500])

    print("\n🤖 Sending to GPT for structured extraction...")
    result = call_openai_for_extraction(ocr_text)

    print("\n📊 Final Extracted Data:")
    print(json.dumps(result, indent=2, ensure_ascii=False))


📄 Running OCR on: testing ocr.pdf

✅ OCR Extracted Text (first 500 chars):
 --- Page 1 ---
Total No. of Questions : 8]

SEAT No. ¢
PD4670 164042176 ce

B.E, (IT)
SOCIAL COMPUTING
(2019 Pattern) Prete VIII) (414451B) (Elective - V)

Time : 2% Hours} os Ww [Max. Marks : 70
Instructions to the candnlatess.- @) wv

D Answer- shor Od Q.3 or 0.4, 0.5 or 0.6, 0.7 or 0.8.

2) Neata lingramns' ‘must be drawn wherever necessary.

3) Fi nts to ther right indicate Sull marks,

4) die suitible data, if necessary.

AX a
ay ,
QI) a) Wiiat is clustering? Explain any clustering al gofit

🤖 Sending to GPT for structured extraction...

📊 Final Extracted Data:
{
  "exam_details": {
    "total_questions": 8,
    "seat_number": "PD4670 164042176",
    "course": "B.E. (IT)",
    "subject": "SOCIAL COMPUTING",
    "pattern": "2019",
    "semester": "VIII",
    "code": "414451B",
    "elective": "V",
    "time": "2.5 Hours",
    "max_marks": 70
  },
  "instructions": [
    "Answer short questions Q.3 or Q.4, Q

In [ ]:
import os
import json
from pdf2image import convert_from_path
import base64
from io import BytesIO
from openai import OpenAI
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.docstore.document import Document
from dotenv import load_dotenv
load_dotenv()

# ========== CONFIG ==========
MODEL = "gpt-4o-mini"
EXTRACTION_PROMPT = """
You are a helpful parser. Extract structured JSON from the following OCR text.
return all data given in pdf 

OCR TEXT:
{OCR_TEXT}
"""
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# Chroma vectorstore directory
CHROMA_DIR = "chroma_db"

# ========== OCR FUNCTION ==========
def ocr_pdf(pdf_path):
    images = convert_from_path(pdf_path)
    ocr_text = ""
    for i, image in enumerate(images):
        # Convert PIL image to base64
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()
        
        # Use OpenAI Vision model to extract text
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Extract all text from this image. Return only the extracted text without any additional formatting or explanations."
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{img_base64}"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=4000,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"[ERROR] OpenAI Vision OCR failed for page {i+1}: {e}")
            text = ""
        
        ocr_text += f"\n--- Page {i+1} ---\n{text}\n"
    return ocr_text.strip()

# ========== GPT FUNCTION ==========
def call_openai_for_extraction(ocr_text, model=MODEL, max_tokens=800):
    prompt = EXTRACTION_PROMPT.replace("{OCR_TEXT}", ocr_text)
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful parser that extracts structured data from OCR'd text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=max_tokens
    )

    output = resp.choices[0].message.content.strip()
    try:
        start = output.find("{")
        end = output.rfind("}") + 1
        json_text = output[start:end]
        parsed = json.loads(json_text)
        parsed["_raw_model_output"] = output
        return parsed
    except Exception as e:
        return {"error": "failed_to_parse_model_output", "raw": output, "exception": str(e)}

# ========== STORE IN CHROMA ==========
def store_in_chroma(data, doc_id="doc1"):
    """
    Store GPT-extracted JSON in ChromaDB as a Document.
    """
    # Convert JSON to string for storage
    content = json.dumps(data, ensure_ascii=False)
    
    embeddings = OpenAIEmbeddings(openai_api_key=API_KEY)
    vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    
    doc = Document(page_content=content, metadata={"source": doc_id})
    vectorstore.add_documents([doc])
    
    # Persist DB to disk
    vectorstore.persist()
    return vectorstore

# ========== RETRIEVE FROM CHROMA ==========
def retrieve_from_chroma(query, top_k=1):
    embeddings = OpenAIEmbeddings(openai_api_key=API_KEY)
    vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    
    results = vectorstore.similarity_search(query, k=top_k)
    return [r.page_content for r in results]

# ========== MAIN ==========
if __name__ == "__main__":
    pdf_path = "testing ocr.pdf"
    print("📄 Running OCR on:", pdf_path)

    ocr_text = ocr_pdf(pdf_path)
    print("\n✅ OCR Extracted Text (first 500 chars):\n", ocr_text[:500])

    print("\n🤖 Sending to GPT for structured extraction...")
    result = call_openai_for_extraction(ocr_text)
    print("\n📊 Final Extracted Data:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

    # Store in Chroma
    print("\n💾 Storing result in ChromaDB...")
    store_in_chroma(result, doc_id=os.path.basename(pdf_path))

    # Retrieve from Chroma
    print("\n🔍 Retrieving from ChromaDB...")
    query = "Extract all data from PDF"
    retrieved_docs = retrieve_from_chroma(query)
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n📄 Retrieved Doc {i}:\n", doc)


📄 Running OCR on: testing ocr.pdf

✅ OCR Extracted Text (first 500 chars):
 --- Page 1 ---
Total No. of Questions : 8]

SEAT No. ¢
PD4670 164042176 ce

B.E, (IT)
SOCIAL COMPUTING
(2019 Pattern) Prete VIII) (414451B) (Elective - V)

Time : 2% Hours} os Ww [Max. Marks : 70
Instructions to the candnlatess.- @) wv

D Answer- shor Od Q.3 or 0.4, 0.5 or 0.6, 0.7 or 0.8.

2) Neata lingramns' ‘must be drawn wherever necessary.

3) Fi nts to ther right indicate Sull marks,

4) die suitible data, if necessary.

AX a
ay ,
QI) a) Wiiat is clustering? Explain any clustering al gofit

🤖 Sending to GPT for structured extraction...

📊 Final Extracted Data:
{
  "exam_details": {
    "total_questions": 8,
    "seat_number": "PD4670 164042176",
    "course": "B.E. (IT)",
    "subject": "SOCIAL COMPUTING",
    "pattern": "2019 Pattern",
    "exam_type": "Prete VIII",
    "code": "414451B",
    "elective": "Elective - V",
    "time": "2.5 Hours",
    "max_marks": 70
  },
  "instructions": [
    "Answer sho

C:\Users\Shree\AppData\Local\Temp\ipykernel_8612\1208756387.py:68: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(openai_api_key=API_KEY)
C:\Users\Shree\AppData\Local\Temp\ipykernel_8612\1208756387.py:69: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)


: 